In [1]:
from fastapi import APIRouter, Query, HTTPException
from typing import Optional, List, Dict, Any, Literal
from datetime import datetime, timedelta
from collections import defaultdict
import pandas as pd
import json
import re
import sys

sys.path.insert(0, r"D:/A0_Project")  # 讓它優先被找到
from PI_SYSTEM.models.sql_db_connect import MySQLConnet

db = MySQLConnet('piaoi_ol_defect_map')
cim_db = MySQLConnet('cim_piaoi')
old_db = MySQLConnet('l6a01_project')

#db = MySQLConnet('piaoi_inspection_density')
#from PI_SYSTEM.models.density.cim_density_job import Config as DensityJobConfig
#from PI_SYSTEM.models.density.API_Config import API_Config

In [2]:
capa_tbns = [tbn for tbn in old_db.list_tables() if 'capa' in tbn]
capa_tbns

2026-04-20 08:38:07,630 - INFO - [list_tables] 成功取得資料表名稱，共 281 張表。


['aoi100_capa_hourly_rawdata',
 'aoi100_capa_hourly_rawdata_fake',
 'aoi100_capa_summary',
 'aoi100_capa_summary_fake',
 'aoi200_capa_hourly_rawdata',
 'aoi200_capa_hourly_rawdata_fake',
 'aoi200_capa_summary',
 'aoi200_capa_summary_fake',
 'aoi300_capa_hourly_rawdata',
 'aoi300_capa_hourly_rawdata_fake',
 'aoi300_capa_summary',
 'aoi300_capa_summary_fake',
 'aoi_summary_aoi300_capa']

In [5]:
capa_tbns

['aoi100_capa_hourly_rawdata',
 'aoi100_capa_summary',
 'aoi200_capa_hourly_rawdata',
 'aoi200_capa_summary',
 'aoi300_capa_hourly_rawdata',
 'aoi300_capa_summary',
 'aoi_summary_aoi300_capa']

In [3]:
old_db.get_runs_between('aoi300_capa_summary', '2026-04-14', '2026-04-16', "run_day")

,aoi,run_day,pi_type,total_glass,target_count,spec,real_day_capa,comment,editor
0,aoi300,2026-04-15,API,112,203.0,90.0,0.551724,,default\n2026-04-15 00:06:04
1,aoi300,2026-04-15,BPI,0,203.0,90.0,0.000000,,default\n2026-04-15 00:06:04
2,aoi300,2026-04-15,ITO,37,203.0,90.0,0.182266,,default\n2026-04-15 00:06:04
3,aoi300,2026-04-15,ALL,149,203.0,90.0,0.733990,,default\n2026-04-15 00:06:04
4,aoi300,2026-04-14,API,146,203.0,90.0,0.719212,,default\n2026-04-14 00:06:05
5,aoi300,2026-04-14,BPI,0,203.0,90.0,0.000000,,default\n2026-04-14 00:06:05
6,aoi300,2026-04-14,ITO,40,203.0,90.0,0.197044,,default\n2026-04-14 00:06:05
7,aoi300,2026-04-14,ALL,186,203.0,90.0,0.916256,,default\n2026-04-14 00:06:05


In [2]:
capa_tbns = [tbn for tbn in old_db.list_tables() if 'capa' in tbn ] #and '_fake' not in tbn
#capa_tb_dict = {}
for tbn in capa_tbns:
    #if '_fake' not in tbn:continue
    #old_db.drop_table(tbn)
    tb = old_db.get_runs_delta_days(tbn, 2, "run_day")
    print(tbn)
    print(tb.iloc[0,:].to_dict())
    #capa_tb_dict[tbn] = tb

2026-04-20 16:48:16,409 - INFO - [list_tables] 成功取得資料表名稱，共 281 張表。


aoi100_capa_hourly_rawdata
{'aoi': 'aoi100', 'run_day': datetime.date(2026, 4, 20), 'hour_int': 11, 'pi_type': 'API', 'hour': 8, 'cumu': 84, 'real_hour_capa': 0.047619047619047616, 'real_cumu_capa': 0.5}
aoi100_capa_hourly_rawdata_fake
{'aoi': 'aoi100', 'run_day': datetime.date(2026, 4, 20), 'hour_int': 11, 'pi_type': 'ALL', 'hour': 8, 'cumu': 84, 'real_hour_capa': 0.047619047619047616, 'real_cumu_capa': 0.5}
aoi100_capa_summary
{'aoi': 'aoi100', 'run_day': datetime.date(2026, 4, 20), 'pi_type': 'API', 'total_glass': 107, 'target_count': 168.0, 'spec': 90.0, 'real_day_capa': 0.6369047619047619, 'comment': '', 'editor': 'default\n2026-04-20 00:06:04'}
aoi100_capa_summary_fake
{'aoi': 'aoi100', 'run_day': datetime.date(2026, 4, 20), 'pi_type': 'API', 'total_glass': 107, 'target_count': 168.0, 'spec': 90.0, 'real_day_capa': 0.6369047619047619, 'comment': '', 'editor': 'default\n2026-04-20 00:06:04'}
aoi200_capa_hourly_rawdata
{'aoi': 'aoi200', 'run_day': datetime.date(2026, 4, 20), 'hour_

In [10]:
'aoi200_capa_hourly_rawdata'
capa_rdf = old_db.get_runs_delta_days('aoi200_capa_summary', 2, "run_day")
capa_rdf

,aoi,run_day,pi_type,total_glass,target_count,spec,real_day_capa,comment,editor
0,aoi200,2026-04-17,API,14,238.0,90.0,0.058824,,default\n2026-04-17 00:06:03
1,aoi200,2026-04-17,BPI,3,238.0,90.0,0.012605,,default\n2026-04-17 00:06:03
2,aoi200,2026-04-17,ALL,17,238.0,90.0,0.071429,,default\n2026-04-17 00:06:03
3,aoi200,2026-04-16,API,89,238.0,90.0,0.373950,,default\n2026-04-16 00:06:03
4,aoi200,2026-04-16,BPI,34,238.0,90.0,0.142857,,default\n2026-04-16 00:06:03
5,aoi200,2026-04-16,ALL,123,238.0,90.0,0.516807,,default\n2026-04-16 00:06:03


In [7]:
capa_sdf = old_db.get_table('aoi_summary_aoi300_capa')
capa_sdf.iloc[-1,:].to_dict()

{'run_day': '2026-04-16',
 'scantime': '2026-04-16 09:45:37',
 'glass_id': 'M36A5LT1H',
 'recipe_id': 'CELL-ITO'}

In [4]:
capa_sdf = old_db.get_table('aoi100_capa_summary')
capa_sdf.iloc[0,:].to_dict()

{'aoi': 'aoi100',
 'run_day': Timestamp('2025-11-01 00:00:00'),
 'pi_type': 'API',
 'total_glass': 67,
 'target_count': 168.0,
 'spec': 90.0,
 'real_day_capa': 0.39880952380952384,
 'comment': '',
 'editor': 'default\n2025-11-26 09:10:11'}

In [6]:
capa_sdf = old_db.get_table('aoi300_capa_summary')
capa_sdf.iloc[0,:].to_dict()
capa_sdf['pi_type'].unique()

array(['API', 'BPI', 'ITO', 'ALL'], dtype=object)

In [5]:
old_rdf = old_db.get_runs_delta_days('aoi200_rawdata_202604', 3, 'scantime')
print(len(old_rdf))
old_sdf = old_db.get_runs_delta_days('aoi_summary_aoi200', 3, 'scantime')
print(len(old_sdf))

175497
669


In [3]:
cim_rdf = cim_db.get_runs_delta_days('cim_defect_202604_aoi200_capic100', 3, 'test_time')
print(len(cim_rdf))
cim_sdf = cim_db.get_runs_delta_days('cim_pi_glass_202604', 3, 'test_time')
print(len(cim_sdf))


22883
1005


In [ ]:
#print(old_rdf.iloc[0].to_dict())
print(old_sdf.iloc[0].to_dict())
print(cim_rdf.iloc[0].to_dict())
print(cim_sdf.iloc[0].to_dict())

{'scantime': Timestamp('2026-04-08 15:15:37'), 'line_id': 'CAAOI202', 'model': 'M215HAN01-T-D-API', 'glass_type': 'M215HAN01-T-D-API', 'recipe_id': '2231', 'glass_id': 'LV6A3WH3A', 'pic_name': 'CaptureImage/RV1_551911_1158498_96', 'x': '551911', 'y': '1158498', 'predict_code': '1', 'judge_code': '1', 'mark': '', 'hour': '', 'dayhour': '', 'day': Timestamp('2026-04-08 15:15:37'), 'year': '', 'month': '', 'season': '', 'week': '', 'yearmonth': '', 'defect_count': '375', 'defect_size': '76', 'open_status': '', 'ai_code_1': '', 'ai_code_2': '', 'ai_code_3': '', 'ai_code_4': '', 'ai_code_5': '', 'gray_name': '', 'ip_num': '', 'first_code': '', 'chip_name': 'D', 'defect_seq': '99', 'image_path': '/CAAOI202/CA001103/LV6A3WH3A/PCS1/20260408151537/', 'over_defect_count': '1', 'large_defect_count': '30', 'middle_defect_count': '297', 'small_defect_count': '47', 'indexid': '0188b283a26c40448e65c72d621e50d4'}
{'run_day': datetime.date(2026, 4, 8), 'scantime': Timestamp('2026-04-08 15:15:37'), 'lin

In [ ]:
db.list_tables()


2026-04-09 15:23:08,703 - INFO - [list_tables] 成功取得資料表名稱，共 1 張表。


['aoi200_202604_api_summary_table']

In [2]:
rows = cim_db.get_runs_delta_days('cim_defect_202604_aoi200_capic100', 2, 'test_time')
rows = rows[rows['sheet_id_chip_id'] =='AF6A3U03A']
rows

,sheet_id_chip_id,chip_id,test_time,defect_size,pox_x1,pox_y1,image_file_path,image_file_name,img_file_url_path,retype_def_code,adc_def_code,pi_time,pi_hour
784,AF6A3U03A,0JE,2026-04-08 14:46:17,S,1279372,417502,Image/CA031401/AF6A3U03A/,RV1_1279372_417502_5.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1446/,nan,PI_Spot_NP,2026-04-08 10:21:11,2026-04-08 09:00:00
785,AF6A3U03A,0EF,2026-04-08 14:46:17,S,714082,59800,Image/CA031401/AF6A3U03A/,RV1_714082_59800_4.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1446/,nan,Not_Found,2026-04-08 10:21:11,2026-04-08 09:00:00
786,AF6A3U03A,0FC,2026-04-08 14:46:17,L,852995,929974,Image/CA031401/AF6A3U03A/,RV1_852995_929974_0.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1446/,nan,Not_Found,2026-04-08 10:21:11,2026-04-08 09:00:00
787,AF6A3U03A,0FC,2026-04-08 14:46:17,M,852823,930090,Image/CA031401/AF6A3U03A/,RV1_852823_930090_1.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1446/,nan,Not_Found,2026-04-08 10:21:11,2026-04-08 09:00:00
788,AF6A3U03A,0LB,2026-04-08 14:46:17,S,1585282,1208619,Image/CA031401/AF6A3U03A/,RV1_1585282_1208619_3.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1446/,nan,Not_Found,2026-04-08 10:21:11,2026-04-08 09:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
893,AF6A3U03A,0KD,2026-04-08 14:43:50,S,1460476,698819,Image/CA031401/AF6A3U03A/,RV1_1460476_698819_94.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1443/,nan,SSIU_Polymer,2026-04-08 10:21:11,2026-04-08 09:00:00
894,AF6A3U03A,0KD,2026-04-08 14:43:50,S,1445916,533188,Image/CA031401/AF6A3U03A/,RV1_1445916_533188_95.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1443/,nan,SSIU_Polymer,2026-04-08 10:21:11,2026-04-08 09:00:00
895,AF6A3U03A,0KF,2026-04-08 14:43:50,S,1391848,188826,Image/CA031401/AF6A3U03A/,RV1_1391848_188826_100.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1443/,nan,SSIU_Polymer,2026-04-08 10:21:11,2026-04-08 09:00:00
896,AF6A3U03A,0CC,2026-04-08 14:43:50,S,341023,803002,Image/CA031401/AF6A3U03A/,RV1_341023_803002_4.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1443/,nan,SSIU_Polymer,2026-04-08 10:21:11,2026-04-08 09:00:00


In [3]:
rows.columns

Index(['sheet_id_chip_id', 'chip_id', 'test_time', 'defect_size', 'pox_x1',
       'pox_y1', 'image_file_path', 'image_file_name', 'img_file_url_path',
       'retype_def_code', 'adc_def_code', 'pi_time', 'pi_hour'],
      dtype='object')

In [20]:
df = db.get_table('aoi200_202604_api_summary_table')
print(len(df))
df.iloc[-1].to_dict()

92


{'id': 92,
 'sheet_id_chip_id': 'YD6A3XD9G',
 'test_time': Timestamp('2026-04-07 07:39:45'),
 'recipe_id': '0251',
 'line_id': 'CAPIC600',
 'aoi': 'aoi200',
 'defect_count': 9,
 'small_defect_count': 5,
 'middle_defect_count': 4,
 'large_defect_count': 0,
 'over_defect_count': 0,
 'pi_time': Timestamp('2026-04-07 05:04:10'),
 'pi_type': 'API',
 'run_day': Timestamp('2026-04-07 00:00:00'),
 'update_time': Timestamp('2026-04-09 15:49:06')}

In [19]:
df[df['sheet_id_chip_id'] =='AF6A3U03A']

,id,sheet_id_chip_id,test_time,recipe_id,line_id,aoi,defect_count,small_defect_count,middle_defect_count,large_defect_count,over_defect_count,pi_time,pi_type,run_day,update_time
14,15,AF6A3U03A,2026-04-08 14:43:50,2250,CAPIC100,aoi200,108,76,31,1,0,2026-04-08 10:21:11,API,2026-04-08,2026-04-09 15:22:29
15,16,AF6A3U03A,2026-04-08 14:46:17,0250,CAPIC100,aoi200,6,3,2,1,0,2026-04-08 10:21:11,API,2026-04-08,2026-04-09 15:22:29


In [3]:
other_db = MySQLConnet('rtms_piaoi_other')
other_db.list_tables()

2026-04-14 10:43:09,346 - INFO - [list_tables] 成功取得資料表名稱，共 2 張表。


['rtms_aoi300_glass_202604', 'rtms_aoi300_raw_202604']

In [4]:
raws = other_db.get_table('rtms_aoi300_raw_202604')
raws['recipe_id'].unique()

array(['B160UAN4A-T-API-20260203', 'B180UAN01-T-API-260405',
       'M270DAN7F-T-API', 'M270QAN06-T-API-260225',
       'M270QAN07-T-API_260410'], dtype=object)